In [ ]:
! pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.enableHiveSupport().getOrCreate()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import time

In [ ]:
# Question 1

In [ ]:
def define_default(database_name):
  spark.sql("USE " + database_name)

def create_database(database_name):
  spark.sql("CREATE DATABASE IF NOT EXISTS " + database_name)
  define_default(database_name)
  spark.sql("SHOW DATABASES").show()

In [ ]:
create_database("pearson")

+---------+
|namespace|
+---------+
|  default|
|  pearson|
+---------+



In [ ]:
# Question 2

In [ ]:
def create_table(schema: str, table_name: str):
  spark.sql("CREATE TABLE IF NOT EXISTS " + table_name + schema)
  spark.sql("SHOW TABLES").show()

In [ ]:
schema_customer="(id String, firstname String, lastname String, gender String, age Int)"

In [ ]:
create_table(schema_customer,"customer")

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  pearson|                case|      false|
|  pearson|            customer|      false|
|  pearson|           follow_up|      false|
|         |df_follow_up_part...|       true|
+---------+--------------------+-----------+



In [ ]:
# Question 3

In [ ]:
def insert_into_table(table_name: str, records: list):

    values = ", ".join(str(record) for record in records)
    query = f"INSERT INTO {table_name} VALUES {values}"
    spark.sql(query)


    spark.sql(f"SELECT * FROM {table_name}").show()

records = [
    ('JD1', 'John', 'Doe', 'M', 42),
    ('JD2', 'Jane', 'Doe', 'F', 36),
    ('KD1', 'Kevin', 'Donut', 'M', 46),
    ('KD2', 'Karen', 'Donut', 'F', 24),
    ('LD1', 'Larry', 'Dawala', 'M', 30),
    ('LD2', 'Lauren', 'Dawala', 'F', 34)
]

insert_into_table("customer", records)


+---+---------+--------+------+---+
| id|firstname|lastname|gender|age|
+---+---------+--------+------+---+
|JD1|     John|     Doe|     M| 42|
|JD2|     Jane|     Doe|     F| 36|
|KD1|    Kevin|   Donut|     M| 46|
|KD2|    Karen|   Donut|     F| 24|
|LD1|    Larry|  Dawala|     M| 30|
|LD2|   Lauren|  Dawala|     F| 34|
+---+---------+--------+------+---+



In [ ]:
# Question 4

In [ ]:

spark.sql("SELECT COUNT(*) AS nb_hommes FROM customer WHERE gender = 'M'").show()
spark.sql("SELECT COUNT(*) AS nb_femmes FROM customer WHERE gender = 'F'").show()


+---------+
|nb_hommes|
+---------+
|        3|
+---------+

+---------+
|nb_femmes|
+---------+
|        3|
+---------+



In [ ]:
# Question 5

In [ ]:
spark.sql("SELECT AVG(age) AS avg_hommes FROM customer WHERE gender = 'M'").show()
spark.sql("SELECT AVG(age) AS avg_femmes FROM customer WHERE gender = 'F'").show()

+------------------+
|        avg_hommes|
+------------------+
|39.333333333333336|
+------------------+

+------------------+
|        avg_femmes|
+------------------+
|31.333333333333332|
+------------------+



In [ ]:
# Question 6

In [ ]:
create_table("(case_id String, customer_id String, item_id String, unit Int, unit_price Int, date String)", "case")

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  pearson|     case|      false|
|  pearson| customer|      false|
+---------+---------+-----------+



In [ ]:
# Question 7

In [ ]:
records = [
    ("('s1', 'jd1', 'it1', 2, 479, '2024-01-01')"),
    ("('s1', 'jd1', 'it2', 1, 899, '2024-01-01')"),
    ("('s2', 'jd2', 'it2', 1, 899, '2024-01-01')"),
    ("('s3', 'kd1', 'it2', 1, 499, '2024-01-01')"),
    ("('s4', 'kd2', 'it3', 1, 1299, '2024-01-02')"),
    ("('s5', 'ld1', 'it2', 2, 879, '2024-01-02')"),
    ("('s6', 'ld2', 'it1', 1, 499, '2024-01-02')"),
    ("('s6', 'ld2', 'it3', 1, 1259, '2024-01-02')")
]
insert_into_table("case", records)

+-------+-----------+-------+----+----------+----------+
|case_id|customer_id|item_id|unit|unit_price|      date|
+-------+-----------+-------+----+----------+----------+
|     s1|        jd1|    it1|   2|       479|2024-01-01|
|     s1|        jd1|    it2|   1|       899|2024-01-01|
|     s2|        jd2|    it2|   1|       899|2024-01-01|
|     s3|        kd1|    it2|   1|       499|2024-01-01|
|     s4|        kd2|    it3|   1|      1299|2024-01-02|
|     s5|        ld1|    it2|   2|       879|2024-01-02|
|     s6|        ld2|    it1|   1|       499|2024-01-02|
|     s6|        ld2|    it3|   1|      1259|2024-01-02|
+-------+-----------+-------+----+----------+----------+



In [ ]:
# Question 8

In [ ]:
def compute_turnover(df: DataFrame, item: str) -> float:
    item_df = df.filter(df['item_id'] == item)
    item_df = item_df.withColumn('turnover', item_df['unit'] * item_df['unit_price'])
    total_turnover = item_df.agg(F.sum('turnover').alias('total_turnover')).collect()[0]['total_turnover']

    return total_turnover

In [ ]:
df_case = spark.sql("SELECT * FROM case")

In [ ]:
df_customer=spark.sql("SELECT * FROM customer")

In [ ]:
compute_turnover(df_case, 'it2')

4055

In [ ]:
# Question 9

In [ ]:
def compute_turnover_for_clients_over_35(df_case: DataFrame, df_customer: DataFrame) -> float:

    df_customer_over_35 = df_customer.filter(df_customer['age'] > 35)
    df_total = df_customer_over_35.join(df_case,
                                        df_customer_over_35.id == F.upper(df_case.customer_id),
                                        how="inner")


    total_turnover = df_total.agg(F.sum(df_total['unit'] * df_total['unit_price'])
                                  .alias("Total turnover for 35+ YO customers")).collect()[0][0]

    return total_turnover

In [ ]:
turnover_over_35 = compute_turnover_for_clients_over_35(df_case, df_customer)

print(f"Le chiffre d'affaires total pour les clients de plus de 35 ans est : {turnover_over_35}")

Le chiffre d'affaires total pour les clients de plus de 35 ans est : 3255


In [ ]:
# Question 10

In [ ]:
create_table( "(id_case String, status String, city String, day String)", "follow_up")

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  pearson|     case|      false|
|  pearson| customer|      false|
|  pearson|follow_up|      false|
+---------+---------+-----------+



In [ ]:
records = [
    "('1','ON GOING', 'NY', 'Day1')",
    "('2','ON GOING', 'DALLAS', 'Day1')",
    "('3','ON GOING', 'LA', 'Day1')",
    "('4','ON GOING', 'NY', 'Day1')",
    "('5','ON GOING', 'DALLAS', 'Day1')",
    "('6','ON GOING', 'LA', 'Day1')",
    "('1','DELIBERATED', 'NY', 'Day2')",
    "('2','DELIBERATED', 'DALLAS', 'Day3')",
    "('3','DELIBERATED', 'LA', 'Day3')",
    "('4','DELIBERATED', 'NY', 'Day4')",
    "('5','DELIBERATED', 'DALLAS', 'Day4')",
    "('6','DELIBERATED', 'LA', 'Day4')",
    "('7','ON GOING', 'NY', 'Day5')",
    "('8','ON GOING', 'DALLAS', 'Day5')",
    "('9','ON GOING', 'LA', 'Day5')",
    "('10','ON GOING', 'NY', 'Day5')",
    "('11','ON GOING', 'DALLAS', 'Day5')",
    "('12','ON GOING', 'LA', 'Day5')",
    "('7','DELIBERATED', 'NY', 'Day6')",
    "('8','DELIBERATED', 'DALLAS', 'Day7')",
    "('9','DELIBERATED', 'LA', 'Day7')",
    "('10','DELIBERATED', 'NY', 'Day8')",
    "('11','DELIBERATED', 'DALLAS', 'Day8')",
    "('12','DELIBERATED', 'LA', 'Day8')"
]

In [ ]:
insert_into_table("follow_up", records)

+-------+-----------+------+----+
|id_case|     status|  city| day|
+-------+-----------+------+----+
|      1|   ON GOING|    NY|Day1|
|      2|   ON GOING|DALLAS|Day1|
|      3|   ON GOING|    LA|Day1|
|      4|   ON GOING|    NY|Day1|
|      5|   ON GOING|DALLAS|Day1|
|      6|   ON GOING|    LA|Day1|
|      1|DELIBERATED|    NY|Day2|
|      2|DELIBERATED|DALLAS|Day3|
|      3|DELIBERATED|    LA|Day3|
|      4|DELIBERATED|    NY|Day4|
|      5|DELIBERATED|DALLAS|Day4|
|      6|DELIBERATED|    LA|Day4|
|      7|   ON GOING|    NY|Day5|
|      8|   ON GOING|DALLAS|Day5|
|      9|   ON GOING|    LA|Day5|
|     10|   ON GOING|    NY|Day5|
|     11|   ON GOING|DALLAS|Day5|
|     12|   ON GOING|    LA|Day5|
|      7|DELIBERATED|    NY|Day6|
|      8|DELIBERATED|DALLAS|Day7|
+-------+-----------+------+----+
only showing top 20 rows



In [ ]:
spark.sql("SELECT day, count(day) as nb FROM follow_up GROUP BY day").show()


+----+---+
| day| nb|
+----+---+
|Day3|  2|
|Day2|  1|
|Day1|  6|
|Day4|  3|
|Day8|  3|
|Day5|  6|
|Day6|  1|
|Day7|  2|
+----+---+



In [ ]:
spark.sql("SELECT CITY, count(CITY) as nb FROM follow_up GROUP BY CITY").show()


+------+---+
|  CITY| nb|
+------+---+
|    LA|  8|
|DALLAS|  8|
|    NY|  8|
+------+---+



In [ ]:
spark.sql("SELECT STATUS, count(STATUS) as nb FROM follow_up GROUP BY STATUS").show()

+-----------+---+
|     STATUS| nb|
+-----------+---+
|   ON GOING| 12|
|DELIBERATED| 12|
+-----------+---+



In [ ]:
# Question 10 = CITY est un excellent choix pour la clé de partitionnement, car elle permet de diviser les données en un nombre restreint de partitions (3 villes seulement), avec une répartition uniforme, tout en optimisant la taille des fichiers dans chaque partition.(HDFS ne supporte pas les petits fichiers)
 #City sera plus optimale que Status parceque pour la question 11 ca sera seulement pour la ville NY.

In [ ]:
cle_optimale="CITY"

In [ ]:
# Question 11

In [ ]:
df_follow_up_resample = spark.sql("SELECT * FROM follow_up")
df_follow_up_resample = df_follow_up_resample.repartition(cle_optimale)

In [ ]:
(df_follow_up_resample
  .coalesce(1)
  .write
  .mode("overwrite")
  .partitionBy(cle_optimale)
  .csv("resample_case")
)


In [ ]:
follow_up_struct = spark.sql("SELECT * FROM follow_up").schema

df_follow_up_partitioned = (
  spark.read
  .schema(follow_up_struct)
  .csv("resample_case")
)


In [ ]:
df_follow_up_partitioned.createOrReplaceTempView("df_follow_up_partitioned")
df_follow_up_partitioned_to_query=spark.sql("SELECT * FROM df_follow_up_partitioned WHERE day=='Day8' AND city=='NY' AND id_case=='10'")

In [ ]:
def mesure_query(df):
  start_time = time.time()
  df.show()
  print(f"Execution time: {time.time() - start_time}")

In [ ]:
mesure_query(df_follow_up_partitioned_to_query)

+-------+-----------+----+----+
|id_case|     status| day|city|
+-------+-----------+----+----+
|     10|DELIBERATED|Day8|  NY|
+-------+-----------+----+----+

Execution time: 0.17524194717407227


In [ ]:
df_follow_up_no_partitioned_to_query=spark.sql("SELECT * FROM follow_up WHERE day=='Day8' AND city=='NY' AND id_case=='10'")

In [ ]:
mesure_query(df_follow_up_no_partitioned_to_query)

+-------+-----------+----+----+
|id_case|     status|city| day|
+-------+-----------+----+----+
|     10|DELIBERATED|  NY|Day8|
+-------+-----------+----+----+

Execution time: 0.27860307693481445


In [ ]:
# En partitionnant les données par CITY, Spark a pu lire uniquement la partition correspondant à NY, réduisant ainsi le volume de données à traiter et améliorant la performance de la requête.